# Lesson 1.1 — Robot state 与 observation

本 notebook 要回答的问题：

> 当环境返回 `obs` 时，它里面究竟包含什么，其中哪一部分是真实机器人真正能够测量到的？

有两个词经常被当作同义词混用，但绝不能混为一谈：

- **state** — 模拟器所知道的关于世界的一切。其中可能包含任何真实传感器都无法提供的量。
- **observation** — 交给 policy 的东西。它是一次*选择*，由 `obs_mode` 参数做出，决定暴露 state 的哪一部分、以什么形状暴露。

按顺序运行各个 cell。有意思的一步是 1.1.4 节，在那里一个关于布局的朴素猜测会被证明是错的。

## 1.1.1 — 创建环境

In [1]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import torch

torch.set_printoptions(precision=4, sci_mode=False)


def make_env(obs_mode="state", control_mode="pd_joint_delta_pos", seed=0):
    env = gym.make(
        "PickCube-v1",
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=1,
    )
    env.reset(seed=seed)
    return env

env = make_env(obs_mode="state")
print(env)
print("observation space:", env.observation_space)
print("action space     :", env.action_space)

2026-09-22 11:36:38,088 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


<TimeLimitWrapper<OrderEnforcing<PickCubeEnv<PickCube-v1>>>>
observation space: Box(-inf, inf, (1, 42), float32)
action space     : Box(-1.0, 1.0, (8,), float32)


## 1.1.2 — 结构化的 observation：`state_dict`

`obs_mode="state"` 返回一个扁平的 tensor，这隐藏了它的结构。同样的信息可以通过 `obs_mode="state_dict"` 以未展开的形式获得，所以先切换模式、看清这棵树，再去信任任何索引。

In [2]:
env_sd = make_env(obs_mode="state_dict")
obs_sd, info = env_sd.reset(seed=0)


def print_tree(data, prefix=""):
    """Print the nested observation structure with tensor shapes."""
    if isinstance(data, dict):
        for key, value in data.items():
            name = f"{prefix}.{key}" if prefix else key
            print_tree(value, name)
    else:
        print(f"  {prefix:<28} shape={getattr(data, 'shape', None)} dtype={getattr(data, 'dtype', None)}")


print("observation tree:")
print_tree(obs_sd)

2026-09-22 11:36:38,466 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


observation tree:
  agent.qpos                   shape=torch.Size([1, 9]) dtype=torch.float32
  agent.qvel                   shape=torch.Size([1, 9]) dtype=torch.float32
  extra.is_grasped             shape=torch.Size([1]) dtype=torch.bool
  extra.tcp_pose               shape=torch.Size([1, 7]) dtype=torch.float32
  extra.goal_pos               shape=torch.Size([1, 3]) dtype=torch.float32
  extra.obj_pose               shape=torch.Size([1, 7]) dtype=torch.float32
  extra.tcp_to_obj_pos         shape=torch.Size([1, 3]) dtype=torch.float32
  extra.obj_to_goal_pos        shape=torch.Size([1, 3]) dtype=torch.float32


## 1.1.3 — 扁平化后的 observation 与 `proprioception`

policy 的输入通常是扁平向量，因此上面的结构必须拼接成一个 tensor。ManiSkill 还通过 `get_proprioception()` 和 `get_state()` 暴露仅与机器人相关的量，借此可以把「机器人对自身的了解」与「任务额外增加的信息」区分开。

In [3]:
obs, info = env.reset(seed=0)
flat = obs[0]
print("flat observation shape:", flat.shape, "dtype:", flat.dtype)
print("first 9 values (joint positions?):", flat[:9])

agent = env.unwrapped.agent


def describe(value, prefix="", depth=0):
    """Describe a nested dict of tensors, since these helpers return nested dicts."""
    if isinstance(value, dict):
        for key, inner in value.items():
            name = f"{prefix}.{key}" if prefix else key
            describe(inner, name, depth + 1)
    else:
        shape = getattr(value, "shape", None)
        print(f"  {prefix:<24} type={type(value).__name__:<10} shape={shape}")


proprio = agent.get_proprioception()
print("\n-- get_proprioception() --")
print("top-level keys:", list(proprio.keys()))
describe(proprio)

state = agent.get_state()
print("\n-- agent.get_state() --")
print("top-level keys:", list(state.keys()))
describe(state)

print("\n-- end-effector --")
print("TCP position:", agent.tcp_pos)
print("TCP pose (7 values, position + quaternion):", agent.tcp_pose.raw_pose[0])

print("\nThese helpers return nested dicts, not flat tensors: there is no single .shape.")

flat observation shape: torch.Size([42]) dtype: torch.float32
first 9 values (joint positions?): tensor([ 0.0353,  0.4007,  0.0196, -1.9187,  0.0374,  2.3366,  0.8044,  0.0400,
         0.0400])

-- get_proprioception() --
top-level keys: ['qpos', 'qvel']
  qpos                     type=Tensor     shape=torch.Size([1, 9])
  qvel                     type=Tensor     shape=torch.Size([1, 9])

-- agent.get_state() --
top-level keys: ['robot_root_pose', 'robot_root_vel', 'robot_root_qvel', 'robot_qpos', 'robot_qvel', 'controller']
  robot_root_pose          type=Pose       shape=torch.Size([1, 7])
  robot_root_vel           type=Tensor     shape=torch.Size([1, 3])
  robot_root_qvel          type=Tensor     shape=torch.Size([1, 3])
  robot_qpos               type=Tensor     shape=torch.Size([1, 9])
  robot_qvel               type=Tensor     shape=torch.Size([1, 9])

-- end-effector --
TCP position: tensor([[0.0123, 0.0380, 0.1822]])
TCP pose (7 values, position + quaternion): tensor([ 0.0123

## 1.1.4 — 预测 42 维布局，然后验证它

**在运行下一个 cell 之前**，先写下你的预测。最直观的猜测是按树打印出来的顺序拼接 `state_dict` 的各个分量：

```text
qpos(9) + qvel(9) + tcp_pose(7) + goal_pos(3) + obj_pose(7)
        + tcp_to_obj_pos(3) + obj_to_goal_pos(3) + is_grasped(1)
```

这个猜测是**错的**。扁平向量的成员顺序与 `state_dict` 的插入顺序不同。把两者对比一下，这个陷阱就变得很具体：如果你靠假设而不是靠验证来索引，下游的每一个 feature 都会被静默地错误标注。

> 在一次实际会话中，这个具体的不一致被直接观察到了：按 `state_dict` 的顺序拼接，再与 `obs_mode="state"` 的结果比较，`np.allclose` 返回了 `False`。

In [4]:
from mani_skill.utils import common


def as_flat(tensor):
    return tensor.reshape(-1).cpu().numpy()


# Candidate A: concatenate in state_dict insertion order
guess_a = np.concatenate([
    as_flat(obs_sd["agent"]["qpos"]),
    as_flat(obs_sd["agent"]["qvel"]),
    as_flat(obs_sd["extra"]["tcp_pose"]),
    as_flat(obs_sd["extra"]["goal_pos"]),
    as_flat(obs_sd["extra"]["obj_pose"]),
    as_flat(obs_sd["extra"]["tcp_to_obj_pos"]),
    as_flat(obs_sd["extra"]["obj_to_goal_pos"]),
    as_flat(obs_sd["extra"]["is_grasped"]),
])

# Candidate B: the layout actually used by obs_mode="state"
guess_b = np.concatenate([
    as_flat(obs_sd["agent"]["qpos"]),
    as_flat(obs_sd["agent"]["qvel"]),
    as_flat(obs_sd["extra"]["is_grasped"]),
    as_flat(obs_sd["extra"]["tcp_pose"]),
    as_flat(obs_sd["extra"]["goal_pos"]),
    as_flat(obs_sd["extra"]["obj_pose"]),
    as_flat(obs_sd["extra"]["tcp_to_obj_pos"]),
    as_flat(obs_sd["extra"]["obj_to_goal_pos"]),
])

flat_state = as_flat(obs[0])
print("state_dict-order matches state:", np.allclose(guess_a, flat_state, atol=1e-6))
print("verified-order  matches state:", np.allclose(guess_b, flat_state, atol=1e-6))

state_dict-order matches state: False
verified-order  matches state: True


### 验证后的 42 维布局

| 切片 | 字段 | 维度 |
|---|---|---:|
| `[0:9]` | `agent.qpos` | 9 |
| `[9:18]` | `agent.qvel` | 9 |
| `[18:19]` | `extra.is_grasped` | 1 |
| `[19:26]` | `extra.tcp_pose` | 7 |
| `[26:29]` | `extra.goal_pos` | 3 |
| `[29:36]` | `extra.obj_pose` | 7 |
| `[36:39]` | `extra.tcp_to_obj_pos` | 3 |
| `[39:42]` | `extra.obj_to_goal_pos` | 3 |

注意 `is_grasped` 的位置：在索引 18，夹在 `qvel` 与 `tcp_pose` 之间，而不是在末尾。这与 `archive/lesson_0_1/verify_state_flattening.py` 和 `build_deployment_safe_observation.py` 中记录的布局相同。

`tcp_pose` 和 `obj_pose` 是 7 维的：3 个位置值，后接一个 4 值 quaternion（分量顺序见 `1.3_coordinate_frames.ipynb`）。

## 1.1.5 — 可部署的 state 与 privileged state

这个 42 维向量混合了两类截然不同的东西：

- **deployable**：关节位置、关节速度、TCP pose。带有 encoder 和 forward kinematics 的真实机械臂能够产生这些量。
- **privileged**：物体位姿、抓取状态以及那些相对向量 —— 模拟器可以精确读取它们，而真实系统通常不借助感知就读不到。

`scripts/pipeline/observation_adapter.py` 把这种划分固化下来。在这里复现它，就能说明为什么一个在全部 42 维上训练出来的 policy 不能原样部署。

In [5]:
import sys
from pathlib import Path

# The observation adapter is a shared module, not part of the conversion pipeline:
# identifying which state fields are deployable is a modelling decision, not a
# conversion step.
from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from observation_adapter import (
    build_deployment_safe_observation,
    build_privileged_observation,
)

deployable = build_deployment_safe_observation(obs_sd)
privileged = build_privileged_observation(obs_sd)

print("deployable dim:", deployable.shape, " = qpos(9) + qvel(9) + tcp_pose(7) + goal_pos(3) = 28")
print("privileged dim:", privileged.shape, " = is_grasped(1) + obj_pose(7) + tcp_to_obj(3) + obj_to_goal(3) = 14")
print("28 + 14 =", deployable.shape[-1] + privileged.shape[-1])
print("\ndeployable excludes:", ["is_grasped", "obj_pose", "tcp_to_obj_pos", "obj_to_goal_pos"])

deployable dim: torch.Size([1, 28])  = qpos(9) + qvel(9) + tcp_pose(7) + goal_pos(3) = 28
privileged dim: torch.Size([1, 14])  = is_grasped(1) + obj_pose(7) + tcp_to_obj(3) + obj_to_goal(3) = 14
28 + 14 = 42

deployable excludes: ['is_grasped', 'obj_pose', 'tcp_to_obj_pos', 'obj_to_goal_pos']


## 小结

1. `obs_mode` 决定 policy 看到什么；底层的 state 更大。
2. 扁平布局**不是** `state_dict` 的插入顺序。要用实验验证 offset；绝不要从打印出的树去推断它们。
3. `proprioception` / `get_state()` 把机器人一侧的量与任务一侧的量区分开。
4. 42 维中包含 14 维模拟器 privileged 信息。任何声称基于 state 的 policy 可部署的说法，都必须回答在真实硬件上用什么替代这 14 个值。

下一节：`1.2_action_space_and_control_modes.ipynb`。